In [26]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env dosyasından API anahtarlarını yükle

# 1. LangSmith (Avrupa Sunucusu) Ayarları
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'nbaapi'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

# Eski sürümlerle uyumluluk için yedek ayarlar
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGCHAIN_PROJECT'] = 'nbaapi'
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')

# 2. Groq API
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [25]:
%pip install dotenv

  Obtaining dependency information for dotenv from https://files.pythonhosted.org/packages/b2/b7/545d2c10c1fc15e48653c91efde329a790f2eecfbbf2bd16003b5db2bab0/dotenv-0.9.9-py2.py3-none-any.whl.metadata
  Obtaining dependency information for python-dotenv from https://files.pythonhosted.org/packages/0d/17/c5c6b53ddc18f297992099b3d9ec16c855c0ccc83263a21fe4d1c625ec6c/python_dotenv-1.2.3-py3-none-any.whl.metadata
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import json
import time
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# 1. Modeli Baslat (120B kotasi taze, guclu ve hizli)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)

# 2. Veriyi Oku
file_path = "../web/data/predictions.json"
with open(file_path, "r", encoding="utf-8") as f:
    players = json.load(f)

# 3. Profesyonel Analist Sablonu (Gercekci & Sifir Halusinasyon)
template = """You are an elite, sharp, and objective NBA Financial and Data Analyst.

Below is the objective breakdown from our trained XGBoost Machine Learning model for this player.
The financial verdict (Overpaid or Underpaid) and the 3 PRIMARY statistical performance drivers that determined this valuation are provided below.

--- PLAYER & AI MODEL BREAKDOWN ---
Name: {isim} ({takim})
Age: {yas}
Actual Contract Salary: ${gercek_maas}
Model Market Valuation: ${tahmin_maas}
Financial Verdict: {status}
The 3 Primary Drivers from the Model: {ana_faktorler}
-----------------------------------

YOUR TASK:
Explain why this player has this valuation ({status}) in 2 to 3 punchy, professional, front-office level sentences.

CRITICAL RULES:
1. Contextual Realism:
   - If the player has "Supermax Contract Ceiling": Acknowledge his elite on-court dominance, but explain that his historic Supermax contract carries a franchise player market premium that exceeds statistical model limits.
   - If the player has "Significant Games Missed to Injury": Explain that the gap is driven by extensive time missed to injury, even though his per-game production remains elite when healthy.
   - If the player is a "Deep Bench Role": Acknowledge that he is on a league-minimum complementary contract and explain his specific supplementary role within limited minutes.
   - Otherwise (Standard Overpaid/Underpaid): Objectively connect the 3 primary drivers to explain the contract value gap.
2. DO NOT quote internal model dollar breakdowns (never say '+$12M' or '-$5M'). Instead, cite the player's actual basketball performance and numbers naturally.
3. Tone: Executive NBA front-office insider writing a concise contract evaluation.

Your Analysis:"""

prompt = PromptTemplate.from_template(template)
rag_chain = prompt | llm

# 4. Tum 380 Oyuncu Icin Dongu (Hazir Olanlari Otomatik Atlar)
total_players = len(players)
print(f"?? Toplam {total_players} oyuncu kontrol ediliyor (haz?r olanlar atlanacak)...")
start_time = time.time()

for i, player in enumerate(players):
    name = player['PLAYER_NAME']
    status = player.get('STATUS', 'Underpaid' if player['PREDICTED_SALARY'] > player['ACTUAL_SALARY'] else 'Overpaid')
    drivers = player.get('KEY_DRIVERS', 'Key statistical factors')
    
    # Zaten analizi varsa saniyede atla
    if "LLM_ANALYSIS" in player and len(player["LLM_ANALYSIS"].strip()) > 20:
        continue
        
    print(f"[{i + 1}/{total_players}] ?? {name} ({status}) analiz ediliyor...")
    
    try:
        cevap = rag_chain.invoke({
            "isim": name,
            "takim": player['TEAM_ABBREVIATION'],
            "yas": player['AGE'],
            "status": status,
            "ana_faktorler": drivers,
            "gercek_maas": format(int(player['ACTUAL_SALARY']), ","),
            "tahmin_maas": format(int(player['PREDICTED_SALARY']), ",")
        })
        player['LLM_ANALYSIS'] = cevap.content.strip()
    except Exception as e:
        print(f"?? Hata olu?tu ({name}): {e}. 5 saniye beklenip tekrar deneniyor...")
        time.sleep(5)
        try:
            cevap = rag_chain.invoke({
                "isim": name,
                "takim": player['TEAM_ABBREVIATION'],
                "yas": player['AGE'],
                "status": status,
                "ana_faktorler": drivers,
                "gercek_maas": format(int(player['ACTUAL_SALARY']), ","),
                "tahmin_maas": format(int(player['PREDICTED_SALARY']), ",")
            })
            player['LLM_ANALYSIS'] = cevap.content.strip()
        except Exception as e2:
            print(f"? Atlaniyor ({name}): {e2}")
            continue

    # Her 20 oyuncuda bir ve sonda ara kaydet
    if (i + 1) % 20 == 0 or (i + 1) == total_players:
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(players, f, indent=4, ensure_ascii=False)
        print(f"?? ---> [KAYDED?LD?] {i + 1} oyuncu dosyaya yaz?ld?.")
        
    time.sleep(1)

# Son kayit
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(players, f, indent=4, ensure_ascii=False)

elapsed = round((time.time() - start_time) / 60, 1)
print(f"\n?? TEBR?KLER! 380 oyuncunun tamam? {elapsed} dakikada ba?ar?yla analiz edilip predictions.json'a kaydedildi.")


?? Toplam 380 oyuncu kontrol ediliyor (haz?r olanlar atlanacak)...
[176/380] ?? KEVIN HUERTER (Overpaid) analiz ediliyor...
[177/380] ?? QUINTEN POST (Overpaid) analiz ediliyor...
[178/380] ?? MAX CHRISTIE (Underpaid) analiz ediliyor...
[179/380] ?? TIDJANE SALAUN (Overpaid) analiz ediliyor...
[180/380] ?? DEANDRE AYTON (Underpaid) analiz ediliyor...
?? ---> [KAYDED?LD?] 180 oyuncu dosyaya yaz?ld?.
[181/380] ?? KELLY OUBRE JR (Underpaid) analiz ediliyor...
[182/380] ?? TRE MANN (Overpaid) analiz ediliyor...
[183/380] ?? TRE JONES (Underpaid) analiz ediliyor...
[184/380] ?? HARRISON BARNES (Overpaid) analiz ediliyor...
[185/380] ?? JEREMIAH FEARS (Overpaid) analiz ediliyor...
[186/380] ?? TAYLOR HENDRICKS (Overpaid) analiz ediliyor...
[187/380] ?? BENNEDICT MATHURIN (Underpaid) analiz ediliyor...
[188/380] ?? JAYLIN WILLIAMS (Underpaid) analiz ediliyor...
[189/380] ?? PAYTON PRITCHARD (Underpaid) analiz ediliyor...
[190/380] ?? GOGA BITADZE (Overpaid) analiz ediliyor...
[191/380] ?? DON

In [ ]:
import pandas as pd
players_df=pd.DataFrame(players)
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PLAYER_NAME        379 non-null    str    
 1   TEAM_ABBREVIATION  379 non-null    str    
 2   AGE                379 non-null    int64  
 3   GP                 379 non-null    int64  
 4   PTS                379 non-null    float64
 5   AST                379 non-null    float64
 6   REB                379 non-null    float64
 7   TOV                379 non-null    float64
 8   ACTUAL_SALARY      379 non-null    int64  
 9   PREDICTED_SALARY   379 non-null    float64
 10  DIFFERENCE         379 non-null    float64
 11  LLM_ANALYSIS       3 non-null      str    
dtypes: float64(6), int64(3), str(3)
memory usage: 35.7 KB
